RISE()/FALL()

24% of dataset contains rise()/fall() operators that are neither supported by STL-SAT or SPOT
--> this notebook is an attempt to better understand the structural implications of the operators and see how processing them could look for our error cases

rise(φ)  →  φ && !(φ)[t-1]     in discrete time
fall(φ)  →  !(φ)[t-1] && φ     in discrete time

yesterday operator (SPOT supported): 
rise(φ)  →  φ_prop & !Yφ_prop      (in SPOT syntax)
fall(φ)  →  !φ_prop & Yφ_prop

why a transformation/time step shift would not work: 


In [1]:
import spot
import pandas as pd
import subprocess

In [2]:
# try different past operator syntaxes for SPOT
tests = [
    'G(Xf -> F(recovery))',          # Next (future)  
    'G(Y(fault) -> F(recovery))',     # Yesterday - no parens
    'G((Yfault) -> F(recovery))',     # Yesterday - no space
    'G((fault & !Xfault) -> F(recovery))',  # X approximation
]
for t in tests:
    try:
        f = spot.formula(t)
        print(f'OK: {t}')
    except Exception as e:
        print(f'FAIL: {t}')
        print(f'     {str(e)[:80]}')

OK: G(Xf -> F(recovery))
FAIL: G(Y(fault) -> F(recovery))
     
>>> G(Y(fault) -> F(recovery))
       ^
syntax error, unexpected opening parent
OK: G((Yfault) -> F(recovery))
OK: G((fault & !Xfault) -> F(recovery))


In [3]:
# Test 1: rise rewritten with Y (exact past operator)
# rise(fault) = fault & Y(!fault)  = fault & !(Yfault)
f1 = spot.formula('G((fault & !Yfault) -> F(recovery))')
f2 = spot.formula('G((fault & !(Yfault)) -> F(recovery))')
print('Test 1 - same formula both ways:', spot.are_equivalent(f1, f2))

# Test 2: rise(fault) vs rise(fault = 1) — your actual use case
# both should expand identically
f3 = spot.formula('G((fault & !Yfault) -> F(recovery))')
f4 = spot.formula('G((fault & !Yfault) -> F(recovery))')
print('Test 2 - identical expansions:', spot.are_equivalent(f3, f4))

# Test 3: confirm Y is truly past — rise should NOT equal just fault
f5 = spot.formula('G((fault & !Yfault) -> F(recovery))')
f6 = spot.formula('G(fault -> F(recovery))')
print('Test 3 - rise != fault (should be False):', spot.are_equivalent(f5, f6))

Test 1 - same formula both ways: True
Test 2 - identical expansions: True
Test 3 - rise != fault (should be False): False


In [4]:
layer1 = pd.read_csv('data/layer1.csv')
skipped = layer1[layer1['suitability'] == 'skip']

In [5]:
for _, row in skipped.iterrows():
    print(f"\n{row['review_id']}")
    print(f"  gold : {row['gold_formula']}")
    print(f"  pred : {row['predicted_formula']}")


MR-003
  gold : G(rise(fault = 1) -> F[0,5](G[0,10](recovery = 1)))
  pred : G(rise(fault) -> F[0,5](rise(recovery) & G[0,10](recovery)))

MR-003
  gold : G(rise(fault = 1) -> F[0,5](G[0,10](recovery = 1)))
  pred : G(rise(fault) -> F[0,5](rise(recovery) & G[0,10](recovery)))

MR-003
  gold : G(rise(fault = 1) -> F[0,5](G[0,10](recovery = 1)))
  pred : G(rise(fault) -> F[0,5](rise(recovery) & G[0,10](recovery)))

MR-008
  gold : G(rise(enable = 1) -> F[0,1](output = 1)) & G(fall(enable = 1) -> F[0,1](output = 0))
  pred : G((rise(enable = 1)) -> F[0,1](rise(output = 1))) & G((fall(enable = 1)) -> F[0,1](fall(output = 1)))

MR-009
  gold : G(rise(enable = 1) -> F[0,1](output = 1)) & G(fall(enable = 1) -> F[0,1](output = 0))
  pred : G(rise(enable) -> F[0,1](output = 1)) & G(fall(enable) -> F[0,1](output = 0))

MR-009
  gold : G(rise(enable = 1) -> F[0,1](output = 1)) & G(fall(enable = 1) -> F[0,1](output = 0))
  pred : G(rise(enable) -> F[0,1](output = 1)) & G(fall(enable) -> F[0,1](ou

In [6]:
result = subprocess.run(
    ['python', 'rise_fall_rewriter.py',
     '--input', 'data/layer1.csv',
     '--output', 'data/rise_fall_ltl.csv'],
    capture_output=True, text=True
)
print(result.stdout)

Rewriting 37 skipped (rise/fall) pairs …

  MR-003    
    gold STL : G(rise(fault = 1) -> F[0,5](G[0,10](recovery = 1)))
    gold LTL : G((fault & !Yfault) -> F(G(recovery)))
    pred STL : G(rise(fault) -> F[0,5](rise(recovery) & G[0,10](recovery)))
    pred LTL : G((fault & !Yfault) -> F((recovery & !Yrecovery) & G(recovery)))

  MR-003    
    gold STL : G(rise(fault = 1) -> F[0,5](G[0,10](recovery = 1)))
    gold LTL : G((fault & !Yfault) -> F(G(recovery)))
    pred STL : G(rise(fault) -> F[0,5](rise(recovery) & G[0,10](recovery)))
    pred LTL : G((fault & !Yfault) -> F((recovery & !Yrecovery) & G(recovery)))

  MR-003    
    gold STL : G(rise(fault = 1) -> F[0,5](G[0,10](recovery = 1)))
    gold LTL : G((fault & !Yfault) -> F(G(recovery)))
    pred STL : G(rise(fault) -> F[0,5](rise(recovery) & G[0,10](recovery)))
    pred LTL : G((fault & !Yfault) -> F((recovery & !Yrecovery) & G(recovery)))

  MR-008    
    gold STL : G(rise(enable = 1) -> F[0,1](output = 1)) & G(fall(enable

In [ ]:
# check directionality of rise/fall vs more relaxed version
# c.contained(f1, f2) checks whether the language of f1 is contained in the language of f2
def implies(f1_str, f2_str):
    c = spot.language_containment_checker()
    return c.contained(spot.formula(f1_str), spot.formula(f2_str))

def classify(gold_ltl, pred_ltl):
    eq   = spot.are_equivalent(gold_ltl, pred_ltl)
    g2p  = implies(gold_ltl, pred_ltl)
    p2g  = implies(pred_ltl, gold_ltl)
    if eq:               return "EQUIVALENT"
    if g2p and not p2g:  return "PRED_STRONGER (gold implies pred, not vice versa)"
    if p2g and not g2p:  return "GOLD_STRONGER (pred implies gold, not vice versa)"
    return "NON_EQUIVALENT"

# MR-003: rise(recovery) in postcondition — the interesting case
gold = "G((fault & !Yfault) -> F(G(recovery)))"
pred = "G((fault & !Yfault) -> F((recovery & !Yrecovery) & G(recovery)))"
print("MR-003:", classify(gold, pred))

# MR-009: rise(enable) vs rise(enable = 1) — should be equivalent
gold = "G((enable & !Yenable) -> F(output)) & G((!enable & Yenable) -> F(!output))"
pred = "G((enable & !Yenable) -> F(output)) & G((!enable & Yenable) -> F(!output))"
print("MR-009:", classify(gold, pred))

# MR-039: gold has no rise, pred has rise — your original question
gold = "G((valve_a & !valve_b) -> F(!valve_a))"
pred = "G(((valve_a & !Yvalve_a) & !valve_b) -> F(!valve_a))"
print("MR-039:", classify(gold, pred))

MR-003: GOLD_STRONGER (pred implies gold, not vice versa)
MR-009: EQUIVALENT
MR-039: PRED_STRONGER (gold implies pred, not vice versa)
